# Per-garden GWAS on the ecotype selection coefficient — GEMMA results

30 gardens × 3 marker classes, each an **independent** GEMMA v0.98.5 univariate LMM
(`-lmm 1`, Wald) on the rank-inverse-normalized per-founder selection coefficient, with a
leave-one-chromosome-out GRM built from all classes pooled.

There is **no cross-site meta** — the all-sites question belongs to the GEA track. So a marker
significant in one garden is a per-garden result, and cross-garden recurrence below is
**descriptive, not a tested contrast**.

Methods of record: `r3_persite_gwas/METHODS_PERSITE_GWAS.md`.


In [1]:

import os, json
import numpy as np, pandas as pd
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 8, "axes.grid": False})

RES = "/global/scratch/projects/fc_moilab/tbellg/kmate/analysis/grenenet_selection/r3_persite_gwas/results/gemma_gwas"
PLOTS = f"{RES}/plots"; os.makedirs(PLOTS, exist_ok=True)
CLASSES = ["snp", "nonsnp", "sv"]
COL = {"snp": "#4C72B0", "nonsnp": "#DD8452", "sv": "#55A868"}

D = {c: np.load(f"{RES}/persite_gwas_{c}.npz", allow_pickle=True) for c in CLASSES}
summary = json.load(open(f"{RES}/persite_gwas_summary.json"))
gardens = np.array(summary["gardens"]); bio1 = np.array(summary["bio1"])
order = np.argsort(bio1)          # cold -> hot, the standard ordering in this project
print(f"{len(gardens)} gardens, bio1 {bio1.min():.1f}-{bio1.max():.1f} C")
for c in CLASSES:
    print(f"  {c:7s} {D[c]['Z'].shape[0]:>9,d} markers x {D[c]['Z'].shape[1]} gardens")


30 gardens, bio1 6.0-21.6 C


  snp     1,752,846 markers x 30 gardens


  nonsnp    525,043 markers x 30 gardens
  sv         12,789 markers x 30 gardens


## 1. Per-garden λ and Bonferroni hit counts

In [2]:

rows = []
for c in CLASSES:
    P = D[c]["P"]; M = P.shape[0]; thr = 0.05 / M
    for si in range(len(gardens)):
        p = P[:, si]; ok = np.isfinite(p)
        rows.append(dict(cls=c, garden=int(gardens[si]), bio1=float(bio1[si]),
                         lam=float(np.median(stats.norm.ppf(1 - p[ok] / 2) ** 2) / stats.chi2.ppf(0.5, 1)),
                         n_bonf=int((p[ok] < thr).sum()),
                         best_nlp=float(-np.log10(np.nanmin(p[ok]))) if ok.any() else np.nan))
tab = pd.DataFrame(rows)
piv = tab.pivot(index="garden", columns="cls", values="n_bonf").reindex(gardens[order])
piv["bio1"] = bio1[order].round(1)
lam = tab.pivot(index="garden", columns="cls", values="lam").reindex(gardens[order]).round(3)
print("Bonferroni hits per garden (cold -> hot):"); print(piv.to_string())
print(f"\nTotal garden-hits: {tab.groupby('cls').n_bonf.sum().to_dict()}")
uniq = {c: int(((D[c]["P"] < 0.05 / D[c]["P"].shape[0]).any(1)).sum()) for c in CLASSES}
print(f"UNIQUE markers hit in >=1 garden: {uniq}")
print(f"\nlambda per garden -- median {lam.median().round(3).to_dict()}")


Bonferroni hits per garden (cold -> hot):
cls     nonsnp  snp  sv  bio1
garden                       
24           0    0   1   6.0
23           0    0   0   7.3
27          17   23   6   7.4
37           0    1   1   8.1
48           1    0   0   8.5
1            1    0   1  10.7
42          10   13   2  10.9
46           0    0   0  11.3
25           4    1   2  11.4
55          31   47   6  11.6
49           0    2   1  11.7
52           2    1   0  12.0
57           0    0   0  12.3
53           0    0   0  12.4
54           4    1   1  12.4
9            2    1   1  12.5
12          18   31   6  13.0
11           5   13   2  13.8
6            0    0   0  15.0
5            0    0   0  15.0
2            0    0   0  15.6
45           0    0   1  15.7
28           3    1   0  16.9
4            0    0   0  17.5
32           0    0   0  17.8
43           0    0   0  18.0
60           9   21   2  19.2
13           2    1   1  20.4
10           4    9   2  20.6
26           0    0   1  21.

UNIQUE markers hit in >=1 garden: {'snp': 112, 'nonsnp': 71, 'sv': 16}

lambda per garden -- median {'nonsnp': 1.015, 'snp': 1.026, 'sv': 1.008}


## 2. QQ per garden — 30 panels, ordered cold → hot

In [3]:

def qq_ax(ax, p, color, label):
    p = p[np.isfinite(p)]; p = np.sort(p)
    n = len(p)
    exp = -np.log10((np.arange(1, n + 1) - 0.5) / n)
    obs = -np.log10(np.clip(p, 1e-300, 1))
    # thin the dense null end so the figure stays light without distorting the tail
    keep = np.concatenate([np.arange(0, min(n, 5000)),
                           np.arange(5000, n, max(1, n // 3000))]) if n > 8000 else np.arange(n)
    ax.plot(exp[keep], obs[keep], ".", ms=1.5, color=color, rasterized=True)
    m = max(exp.max(), obs.max())
    ax.plot([0, m], [0, m], "-", lw=0.6, color="0.5")
    lamv = np.median(stats.norm.ppf(1 - p / 2) ** 2) / stats.chi2.ppf(0.5, 1)
    ax.annotate(f"{label}\n$\\lambda$={lamv:.2f}", xy=(0.04, 0.96), xycoords="axes fraction",
                va="top", ha="left", fontsize=6.5)
    ax.set_xticks([]); ax.set_yticks([])

for c in CLASSES:
    P = D[c]["P"]
    fig, axes = plt.subplots(5, 6, figsize=(13, 10.5), sharex=True, sharey=True)
    for k, si in enumerate(order):
        qq_ax(axes.flat[k], P[:, si], COL[c], f"g{gardens[si]} ({bio1[si]:.0f}C)")
    for ax in axes.flat[len(order):]:
        ax.axis("off")
    fig.supxlabel("expected $-\\log_{10}p$"); fig.supylabel("observed $-\\log_{10}p$")
    fig.tight_layout()
    fig.savefig(f"{PLOTS}/qq_grid_{c}.png", dpi=140, bbox_inches="tight")
    print(f"wrote qq_grid_{c}.png  (30 gardens, cold->hot)")
    plt.close(fig)


wrote qq_grid_snp.png  (30 gardens, cold->hot)


wrote qq_grid_nonsnp.png  (30 gardens, cold->hot)


wrote qq_grid_sv.png  (30 gardens, cold->hot)


## ⚠️ Read the MAC-stratified QQ before interpreting any hit

Genomic-control λ is ~1.0 in every garden and every MAC stratum, which looks like clean
calibration. It is not sufficient: λ is a **median** statistic and never touches the extreme
quantiles that Bonferroni hits are drawn from. Stratified by MAC, the tail runs **5–60×** hot,
and **90% of significant SNP markers sit in MAC 5–11**, the worst-calibrated stratum.

Not all artifact — a 3–4× excess at p<1e-4 persists in clean strata (expected for a polygenic
trait plus LD), and the inflation is non-monotone, rising again at MAC ≥46 where power says real
signal belongs. Rare-end artifact superimposed on common-end signal; these numbers cannot
separate them. **Open decision: permutation null per stratum, or raise the MAC floor.**


## 3. QQ by MAC stratum — the decision panel

In [4]:

# THE decision panel: QQ split by MAC stratum. lambda is ~1.0 in every stratum while the tail
# runs 5-60x hot -- lambda is a median statistic and never touches the quantiles hits live in.
mc = pd.read_csv(f"{RES}/mac_calibration.csv")
mac = {c: np.load(f"{RES}/mac_by_class.npz")[c] for c in CLASSES}
BINS = [(5, 7), (8, 11), (12, 22), (23, 45), (46, 10**9)]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for ax, c in zip(axes, CLASSES):
    P = D[c]["P"]; m_all = mac[c]
    for (lo, hi), shade in zip(BINS, np.linspace(0.15, 0.85, len(BINS))):
        sel = (m_all >= lo) & (m_all <= hi)
        if sel.sum() < 100:
            continue
        p = P[sel].ravel(); p = p[np.isfinite(p)]
        n = len(p); p = np.sort(p)
        exp = -np.log10((np.arange(1, n + 1) - 0.5) / n)
        obs = -np.log10(np.clip(p, 1e-300, 1))
        keep = np.concatenate([np.arange(0, min(n, 4000)),
                               np.arange(4000, n, max(1, n // 2500))]) if n > 6000 else np.arange(n)
        lab = f"MAC {lo}-{hi}" if hi < 10**9 else f"MAC$\\geq${lo}"
        ax.plot(exp[keep], obs[keep], ".", ms=1.5, rasterized=True,
                color=plt.cm.viridis(shade), label=lab)
    m = 7.5
    ax.plot([0, m], [0, m], "-", lw=0.7, color="0.4")
    ax.annotate(c, xy=(0.04, 0.96), xycoords="axes fraction", va="top", fontsize=9, weight="bold")
    ax.set_xlabel("expected $-\\log_{10}p$"); ax.set_xlim(0, m)
    ax.legend(fontsize=6, loc="lower right", markerscale=4, frameon=False)
axes[0].set_ylabel("observed $-\\log_{10}p$")
fig.tight_layout(); fig.savefig(f"{PLOTS}/qq_by_mac_stratum.png", dpi=150, bbox_inches="tight")
print("wrote qq_by_mac_stratum.png")
plt.close(fig)
print(mc[["cls", "mac_lo", "mac_hi", "lambda_gc", "ratio_1e4", "ratio_1e6"]].to_string(index=False))


wrote qq_by_mac_stratum.png
   cls  mac_lo  mac_hi  lambda_gc  ratio_1e4  ratio_1e6
   snp       5       7   0.936080   4.669942  59.506182
   snp       8      11   1.011216   4.036259  24.566470
   snp      12      22   1.029891   2.863553  10.201846
   snp      23      45   1.062270   3.754768   5.611609
   snp      46     115   1.084347   3.903451  18.937452
nonsnp       5       7   0.940011   5.197216  51.146199
nonsnp       8      11   1.002364   4.010166  30.909184
nonsnp      12      22   1.014210   2.862124  11.378473
nonsnp      23      45   1.042522   3.297091   8.226274
nonsnp      46     115   1.074611   3.554935  21.944045
    sv       5       7   0.964816   5.884544  52.141527
    sv       8      11   0.988820   4.399586  25.879917
    sv      12      22   1.030071   5.579912 119.569550
    sv      23      45   1.079449   2.388760   0.000000
    sv      46     115   1.062220   4.329090   0.000000


## 4. Manhattan per garden — 30 panels, ordered cold → hot

In [5]:

CH = ["chr1", "chr2", "chr3", "chr4", "chr5"]
def genome_coords(chrom, pos):
    off, cum = {}, 0
    for c in CH:
        off[c] = cum; cum += pos[chrom == c].max() if (chrom == c).any() else 0
    x = np.array([off[c] for c in chrom]) + pos
    centers = [off[c] + (pos[chrom == c].max() / 2 if (chrom == c).any() else 0) for c in CH]
    return x, centers, cum

for c in CLASSES:
    chrom = D[c]["chrom"].astype(str); pos = D[c]["pos"].astype(float); P = D[c]["P"]
    x, centers, tot = genome_coords(chrom, pos)
    band = np.isin(chrom, CH[::2])
    thr = -np.log10(0.05 / P.shape[0])
    fig, axes = plt.subplots(10, 3, figsize=(14, 18), sharex=True, sharey=True)
    for k, si in enumerate(order):
        ax = axes.flat[k]
        nlp = -np.log10(np.clip(P[:, si], 1e-300, 1))
        # keep every informative point, thin the dense null floor 20x -- plotting all ~550k
        # sub-threshold markers x 30 panels is 16M glyphs per figure and renders nothing extra
        strong = nlp > 2.0
        floor = np.where(nlp <= 2.0)[0][::20]
        keep = np.zeros(len(nlp), bool); keep[strong] = True; keep[floor] = True
        ax.scatter(x[keep & band], nlp[keep & band], s=0.7, c="0.62", rasterized=True, lw=0)
        ax.scatter(x[keep & ~band], nlp[keep & ~band], s=0.7, c=COL[c], rasterized=True, lw=0)
        ax.axhline(thr, color="crimson", lw=0.6, ls="--")
        nb = int((P[:, si] < 0.05 / P.shape[0]).sum())
        ax.annotate(f"g{gardens[si]}  {bio1[si]:.0f}$^\\circ$C  n={nb}", xy=(0.01, 0.94),
                    xycoords="axes fraction", va="top", fontsize=6.5)
        ax.set_xticks([]); ax.set_ylim(0.5, max(thr + 1.5, np.nanmax(nlp) * 1.05))
    for ax in axes.flat[len(order):]:
        ax.axis("off")
    fig.supylabel("$-\\log_{10}p$"); fig.supxlabel("genome position (chr1-chr5)")
    fig.tight_layout()
    fig.savefig(f"{PLOTS}/manhattan_grid_{c}.png", dpi=130, bbox_inches="tight")
    print(f"wrote manhattan_grid_{c}.png")
    plt.close(fig)


wrote manhattan_grid_snp.png


wrote manhattan_grid_nonsnp.png


wrote manhattan_grid_sv.png


## 5. The hit list, with MAC attached

In [6]:

# the actual hit list, with MAC attached -- the column that decides whether to trust a row
recs = []
for c in CLASSES:
    P = D[c]["P"]; thr = 0.05 / P.shape[0]
    chrom = D[c]["chrom"].astype(str); pos = D[c]["pos"]
    m_all = mac[c]
    hit_i, hit_g = np.where(P < thr)
    for i, g in zip(hit_i, hit_g):
        recs.append(dict(cls=c, chrom=chrom[i], pos=int(pos[i]), garden=int(gardens[g]),
                         bio1=round(float(bio1[g]), 1), mac=int(m_all[i]),
                         maf=round(m_all[i] / 231, 4),
                         nlp=round(float(-np.log10(max(P[i, g], 1e-300))), 2)))
hits = pd.DataFrame(recs).sort_values("nlp", ascending=False)
hits.to_csv(f"{RES}/persite_bonferroni_hits.csv", index=False)
print(f"{len(hits)} garden-hits; {hits.groupby('cls').size().to_dict()}")
print(f"\nMAC distribution of hits vs panel:")
for c in CLASSES:
    h = hits[hits.cls == c]
    print(f"  {c:7s} hits MAC<12: {(h.mac<12).mean():5.1%}  |  panel MAC<12: {(mac[c]<12).mean():5.1%}")
print("\nTop 15 by -log10 p:")
print(hits.head(15).to_string(index=False))


316 garden-hits; {'nonsnp': 113, 'snp': 166, 'sv': 37}

MAC distribution of hits vs panel:
  snp     hits MAC<12: 91.6%  |  panel MAC<12: 36.6%
  nonsnp  hits MAC<12: 79.6%  |  panel MAC<12: 35.8%
  sv      hits MAC<12: 54.1%  |  panel MAC<12: 55.1%

Top 15 by -log10 p:
   cls chrom      pos  garden  bio1  mac    maf   nlp
   snp  chr2 13900692      27   7.4    8 0.0346 10.83
   snp  chr4 18399380      55  11.6    5 0.0216 10.68
nonsnp  chr5   482054      12  13.0    5 0.0216 10.10
   snp  chr5 16190357      12  13.0    5 0.0216  9.98
   snp  chr5 16190666      12  13.0    5 0.0216  9.88
   snp  chr4  1254136      55  11.6    5 0.0216  9.66
nonsnp  chr2 16378782      12  13.0    5 0.0216  9.51
   snp  chr4 15204084      55  11.6    5 0.0216  9.29
   snp  chr5 19620565      55  11.6    5 0.0216  9.27
   snp  chr4 12170010      12  13.0   10 0.0433  9.18
   snp  chr5  3161591      42  10.9    5 0.0216  9.12
   snp  chr5  3163105      42  10.9    5 0.0216  9.12
   snp  chr1  4707836      